# Lesson 04: Pretraining Objectives

Goal: understand what loss functions teach language models during pretraining.

## Learning goals

- Explain causal language modeling (next-token prediction)
- Compare CLM with masked language modeling (MLM)
- Compute cross-entropy loss on toy logits
- Understand perplexity at a high level

## Causal language modeling (CLM)

For each position, predict the next token using only previous tokens.

If tokens are `t_1, t_2, ..., t_n`, we maximize:

`sum_i log p(t_i | t_<i)`

This is the standard objective for GPT-style models.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

def softmax(x, axis=-1):
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x_shift)
    return e / np.sum(e, axis=axis, keepdims=True)

def cross_entropy_from_logits(logits, targets):
    probs = softmax(logits, axis=-1)
    n = targets.shape[0]
    p_true = probs[np.arange(n), targets]
    loss = -np.log(p_true + 1e-12).mean()
    return loss, probs

In [ ]:
# Toy next-token example
vocab = ['I', 'love', 'LLMs', '.', '<eos>']
token_to_id = {t: i for i, t in enumerate(vocab)}

# Input sequence: I love LLMs
# Targets for next-token prediction: love, LLMs, .
targets = np.array([token_to_id['love'], token_to_id['LLMs'], token_to_id['.']])

# Model logits for each position (3 positions x 5 vocab tokens)
logits = np.array([
    [0.1, 2.0, 0.3, -0.2, -1.0],
    [0.0, 0.2, 1.8, 0.1, -0.5],
    [-0.2, 0.1, 0.3, 1.6, -0.3],
])

loss, probs = cross_entropy_from_logits(logits, targets)
print('Probabilities:
', probs)
print('Cross-entropy loss:', round(loss, 4))
print('Perplexity:', round(float(np.exp(loss)), 4))

Lower loss means higher probability assigned to the correct next tokens.
Perplexity is `exp(loss)` for this token-level negative log-likelihood setup.

## Masked language modeling (MLM)

BERT-style training masks some tokens and predicts only those masked positions.

Example:
- Original: `Paris is the capital of France`
- Masked: `Paris is the [MASK] of France`
- Predict: `capital`

Key difference:
- CLM: left-to-right prediction
- MLM: bidirectional context around masked tokens

In [ ]:
# Tiny MLM loss example on masked positions only
masked_positions = np.array([0, 2])
masked_targets = np.array([1, 3])
masked_logits = np.array([
    [0.0, 1.5, 0.2, -0.4, 0.1],
    [0.2, -0.1, 0.3, 1.2, 0.0],
])

mlm_loss, _ = cross_entropy_from_logits(masked_logits, masked_targets)
print('Masked positions:', masked_positions)
print('MLM loss:', round(mlm_loss, 4))

## Checkpoints

1. Why is causal masking required for CLM?
2. Why can MLM use both left and right context?
3. What does perplexity tell us, and what does it not tell us?

## Next directions

You can now branch into:
- Tokenization (BPE/unigram)
- Scaling laws
- Finetuning and instruction tuning
- Evaluation and prompting